# Semaine 3 - Jour 3 : Implémentation Avancée d'un Serveur MCP (Model Context Protocol)

## 🎯 Objectifs pédagogiques
- Maîtriser le cycle de vie d'un **MCP Server** et sa communication avec un **MCP Client**.
- Implémenter et exposer les trois piliers du protocole : **Tools**, **Resources**, et **Prompts**.
- Comprendre la découverte dynamique, la validation stricte des paramètres via **Pydantic** et la gestion des erreurs.
- Intégrer les patterns de sécurité, gouvernance et d'architecture d'entreprise adaptés aux profils Tech Lead / Architecte.
- Préparer l'interconnexion avec les SDK d'orchestration du marché (OpenAI Agents SDK, LangGraph, etc.).

---

## 1. Foundations & Vision Architecture

### 1.1 Présentation détaillée de MCP
Le protocole **Model Context Protocol (MCP)** standardise la couche de communication entre les LLM (via leurs applications hôtes ou orchestrateurs) et les systèmes d'information (bases de données, API, systèmes de fichiers, SaaS).

En architecture logicielle traditionnelle, l'intégration d'outils externes (Function Calling) souffre d'un couplage fort. Chaque framework d'agents (LangChain, AutoGen, CrewAI, OpenAI SDK) possède sa propre structure de déclaration de fonctions. MCP introduit une couche d'abstraction ouverte et agnostique.

### 1.2 Comparaison : API Classique vs Architecture MCP

#### Approche API Classique (Point-à-Point)
- **Flot** : L'orchestrateur de l'application IA doit charger manuellement les définitions OpenAPI/Swagger, formater le JSON-Schema pour le modèle spécifique, intercepter l'appel du modèle, puis exécuter localement le code de glue pour appeler l'API.
- **Inconvénient** : Maintenance lourde, couplage fort entre le moteur d'exécution de l'IA et les systèmes sources, duplication des configurations de sécurité.

#### Approche MCP (Découplage par Protocole)
- **Flot** : L'application IA instancie un unique **MCP Client**. Ce client se connecte à un ou plusieurs **MCP Servers** via un canal standardisé (Stdio ou SSE). Par introspection (méthode `list_tools`), le serveur dicte ses capacités en temps réel au client.
- **Inconvénient** : Légère surcouche réseau/processus (compensée par le gain de maintenabilité).

### 1.3 Architecture d'Entreprise, Sécurité et Gouvernance

D'un point de vue architectural, le déploiement de serveurs MCP à l'échelle d'une entreprise soulève des enjeux critiques :

1. **Isolation des Processus (Sandboxing)** : Un serveur MCP s'exécute de manière autonome. En mode `Stdio`, il tourne dans son propre sous-processus de l'hôte. En mode `SSE` (Server-Sent Events), il s'apparente à un microservice conteneurisé. Cela évite qu'une faille dans le code d'un connecteur ne compromette l'application IA principale.
2. **Gouvernance des Accès (RBAC & Audit)** : Le serveur MCP sert de passerelle. Il valide que l'intention de l'IA ou l'identité de l'utilisateur final sous-jacent possède les habilitations nécessaires avant de requêter la base de données ou le CRM. Toutes les actions (appels d'outils, accès aux ressources) génèrent des logs d'audit standardisés.
3. **Transports Supportés** :
   - **Stdio** : Idéal pour les agents locaux, les CLI, les extensions d'IDE (ex: Claude Desktop). La communication se fait par entrées/sorties standard cryptées par le canal parent.
   - **SSE / HTTP** : Recommandé pour l'architecture microservices d'entreprise. Permet le load-balancing, le déploiement sur Kubernetes et l'exposition d'outils partagés à l'échelle de l'organisation.

## 2. Les Trois Piliers du Protocole MCP

Le protocole s'articule autour de trois concepts fondamentaux qu'un serveur peut exposer :

| Concept | Rôle | Mode d'interaction LLM | Exemple concret |
| :--- | :--- | :--- | :--- |
| **Resources** | Données en lecture seule, typées (MIME). | L'IA les lit pour enrichir son contexte (RAG, logs). | Fichier de config, schéma DB, logs système. |
| **Tools** | Fonctions exécutables avec effets de bord. | L'IA decides de les appeler avec des arguments validés. | `execute_sql`, `send_slack`, `create_ticket`. |
| **Prompts** | Modèles de prompts réutilisables et paramétrés. | Proposés à l'utilisateur ou à l'agent pour guider la tâche. | `code_review`, `bug_analysis_template`. |

MCP

├── Tools

├── Resources

└── Prompts

### Resources

Une Resource n'est pas une action.
C'est une donnée.

Exemples :
```
Documentation RH
Wiki interne
PDF
Base de connaissance
Configuration
```
Le LLM lit. Il ne déclenche aucune action.

### Exemple

Documentation interne :
Politique de congés

Tout congé supérieur à 5 jours
doit être validé par le manager.

Avec MCP: 
```
resources = {
    "policy://vacation":
        "Tout congé supérieur à 5 jours..."
}
```

### Pourquoi c'est différent d'un Tool ?

Avec un Tool :
```
search_policy()
```
on effectue une action.

Avec une Resource :
```
policy://vacation
```
on lit directement une donnée.

### Architecture
```
LLM
 ↓
Resource
 ↓
Document
```

### Exemple de code Resource

In [ ]:
class Resource:

    def __init__(
        self,
        uri,
        content
    ):
        self.uri = uri
        self.content = content

class MCPServer:

    def __init__(self):

        self.resources = {}

    def register_resource(
        self,
        uri,
        content
    ):
        self.resources[uri] = content

    def read_resource(
        self,
        uri
    ):
        return self.resources[uri]



### Utilisation 

In [4]:
server.register_resource(
    "policy://vacation",
    """
    Les congés supérieurs à 5 jours
    nécessitent une validation.
    """
)

NameError: name 'server' is not defined

### Lecture :

In [ ]:
server.read_resource(
    "policy://vacation"
)

## Prompts

C'est la partie la moins comprise.

Un Prompt est un template réutilisable.

Exemple :
```
Résume le document suivant :

{{document}}
```
Ou :
```
Analyse ce contrat et identifie :

- les risques
- les obligations
- les pénalités

Contrat :

{{contract}}
```
Un Prompt n'est :

ni une action
ni une donnée

C'est une recette.

Architecture
```
LLM
 ↓
Prompt
 ↓
Template
 ↓
Prompt final
```

In [5]:
class MCPServer:

    def __init__(self):

        self.prompts = {}

    def register_prompt(
        self,
        name,
        template
    ):
        self.prompts[name] = template

    def get_prompt(
        self,
        name
    ):
        return self.prompts[name]

# Enregistrement :

server.register_prompt(
    "summarize",
    """
    Résume le document suivant :

    {document}
    """
)

# Utilisation :

template = server.get_prompt(
    "summarize"
)

prompt = template.format(
    document="..."
)

NameError: name 'server' is not defined

## 🛠️ 3. Implémentation Progressive du Serveur MCP

Pour cette démonstration et pour garantir la portabilité au sein du notebook, nous allons utiliser le SDK Python officiel `mcp`. Nous allons construire un serveur d'entreprise capable de manipuler une base de données d'incidents fictive.

### Étape 3.1 : Installation des dépendances et initialisation du stockage

In [ ]:
# Installation du SDK MCP officiel et de Pydantic pour la validation des données
%pip install mcp pydantic --quiet

In [ ]:
import asyncio
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from mcp import Server
import mcp.types as types

# Base de données d'incidents d'entreprise simulée en mémoire pour notre serveur MCP
INCIDENT_DATABASE = {
    "INC-001": {"id": "INC-001", "title": "Panne de la passerelle de paiement Java", "severity": "CRITICAL", "status": "OPEN", "assignee": "TechLead-Java"},
    "INC-002": {"id": "INC-002", "title": "Erreur de validation de schéma Karate test", "severity": "MAJOR", "status": "IN_PROGRESS", "assignee": "Architect-AI"},
    "INC-003": {"id": "INC-003", "title": "Fuite de mémoire Spring Boot 4 custom starter", "severity": "CRITICAL", "status": "OPEN", "assignee": "None"}
}

print(f"Base de données initialisée avec {len(INCIDENT_DATABASE)} incidents fictifs.")

### Étape 3.2 : Définition des schémas de validation (Pydantic)

En tant que développeur backend senior, vous savez que la validation aux frontières est non-négociable. Lorsque l'IA génère les arguments d'un outil, ces arguments peuvent être mal formés ou incomplets. Nous utilisons des modèles Pydantic pour valider strictement les entrées générées par le modèle.

In [ ]:
class CreateIncidentArgs(BaseModel):
    title: str = Field(..., description="Titre explicite de l'incident technique détecté.")
    severity: str = Field(..., description="Niveau de criticité. Valeurs autorisées : MINOR, MAJOR, CRITICAL.")
    assignee: Optional[str] = Field("None", description="Nom ou rôle du membre de l'équipe assigné.")

    class Config:
        extra = "forbid" # Interdiction stricte de paramètres fantômes générés par le LLM

### Étape 3.3 : Instanciation du Serveur MCP et Enregistrement des Capacités

Nous allons initialiser l'instance principale du serveur et déclarer explicitement nos outils, ressources et invites (prompts). Le protocole repose sur des mécanismes d'enregistrement asynchrones.

In [ ]:
# Création du serveur MCP nommé 'Enterprise-Support-Server'
enterprise_mcp_server = Server("Enterprise-Support-Server")

## -------------------------------------------------------------------------
## PILLIER 1 : ENREGISTREMENT ET GESTION DES OUTILS (TOOLS)
## -------------------------------------------------------------------------

@enterprise_mcp_server.list_tools()
async def handle_list_tools() -> List[types.Tool]:
    """
    Découverte dynamique des outils : Le serveur renvoie la liste des outils disponibles 
    et leurs schémas JSON attendus au client MCP.
    """
    return [
        types.Tool(
            name="get_all_incidents",
            description="Récupère la liste complète des incidents ouverts au support technique.",
            inputSchema={
                "type": "object",
                "properties": {}
            }
        ),
        types.Tool(
            name="create_new_incident",
            description="Crée un nouvel incident dans la base de données de support après validation.",
            inputSchema=CreateIncidentArgs.model_json_schema() # Liaison directe avec le schéma Pydantic
        )
    ]

### Étape 3.4 : Implémentation de la Logique d'Exécution des Outils

La méthode `call_tool` centrale route l'exécution vers la logique métier correspondante en fonction du nom de l'outil invoqué par l'IA.

In [ ]:
@enterprise_mcp_server.call_tool()
async def handle_call_tool(name: str, arguments: Dict[str, Any]) -> List[types.TextContent]:
    """
    Exécute l'outil demandé par le LLM après validation stricte des arguments.
    """
    try:
        if name == "get_all_incidents":
            import json
            return [types.TextContent(type="text", text=json.dumps(INCIDENT_DATABASE, indent=2))]
            
        elif name == "create_new_incident":
            # Validation à l'aide du modèle Pydantic défini précédemment
            validated_args = CreateIncidentArgs(**arguments)
            
            # Génération d'un ID unique synchrone pour la démo
            new_id = f"INC-00{len(INCIDENT_DATABASE) + 1}"
            
            new_incident = {
                "id": new_id,
                "title": validated_args.title,
                "severity": validated_args.severity.upper(),
                "status": "OPEN",
                "assignee": validated_args.assignee
            }
            
            # Mutation d'état de la couche persistance
            INCIDENT_DATABASE[new_id] = new_incident
            
            return [types.TextContent(
                type="text", 
                text=f"Succès : Incident {new_id} correctement enregistré dans le système."
            )]
        else:
            raise ValueError(f"Outil inconnu : {name}")
            
    except Exception as e:
        # Gestion des erreurs robuste pour éviter le crash du sous-processus MCP
        return [types.TextContent(type="text", text=f"Erreur d'exécution de l'outil [{name}]: {str(e)}")]

### Étape 3.5 : Implémentation des Ressources (Resources) et Prompts

Exposons maintenant une ressource statique de gouvernance (la charte de traitement des tickets) et un modèle d'invite pour aider le LLM à formuler des synthèses de crise.

In [ ]:
## -------------------------------------------------------------------------
## PILLIER 2 : ENREGISTREMENT ET LECTURE DES RESSOURCES
## -------------------------------------------------------------------------

@enterprise_mcp_server.list_resources()
async def handle_list_resources() -> List[types.Resource]:
    return [
        types.Resource(
            uri="enterprise://policies/severity-rules",
            name="Charte de Sécurité et SLA des Incidents",
            description="Règles de gouvernance pour déterminer les niveaux d'urgence des anomalies.",
            mimeType="text/plain"
        )
    ]

@enterprise_mcp_server.read_resource()
async def handle_read_resource(uri: str) -> str:
    if uri == "enterprise://policies/severity-rules":
        return "CRITICAL : Interruption totale de service. SLA de résolution = 2h.\nMAJOR : Dégradation forte sans contournement. SLA = 8h."
    raise ValueError("Ressource introuvable")

## -------------------------------------------------------------------------
## PILLIER 3 : ENREGISTREMENT DES PROMPTS MUTALISÉS
## -------------------------------------------------------------------------

@enterprise_mcp_server.list_prompts()
async def handle_list_prompts() -> List[types.Prompt]:
    return [
        types.Prompt(
            name="incident_briefing",
            description="Génère un canevas de rapport de crise officiel pour la direction.",
            arguments=[
                types.PromptArgument(
                    name="manager_name", 
                    description="Nom du responsable à qui est destiné la note.", 
                    required=True
                )
            ]
        )
    ]

@enterprise_mcp_server.get_prompt()
async def handle_get_prompt(name: str, arguments: Dict[str, str]) -> types.GetPromptResult:
    if name == "incident_briefing":
        manager = arguments.get("manager_name", "Directeur")
        return types.GetPromptResult(
            description="Template de briefing",
            messages=[
                types.PromptMessage(
                    role="user",
                    content=types.TextContent(
                        type="text",
                        text=f"Bonjour {manager}, veuillez analyser l'état des incidents CRITICAL actuellement ouverts et en faire un résumé exécutif selon nos règles de SLA."
                    )
                )
            ]
        )
    raise ValueError("Prompt introuvable")

## 4. Simulation d'un Client MCP Introspectif

Pour valider notre serveur sans lancer de processus OS complexe externe, nous allons simuler le comportement d'un **MCP Client** (comme le ferait l'application hôte OpenAI ou Claude) qui interroge directement les gestionnaires asynchrones du serveur.

In [ ]:
async def simulate_client_discovery_and_execution():
    print("=== SIMULATION DU CLIENT MCP : PHASE DE DÉCOUVERTE ===\n")
    
    # 1. Le client demande la liste des outils exposés par le serveur
    discovered_tools = await handle_list_tools()
    for tool in discovered_tools:
        print(f"[Outil Découvert] Name: {tool.name}")
        print(f"  Description: {tool.description}")
        print(f"  JSON Schema de validation des inputs: {tool.inputSchema}\n")
        
    print("=== SIMULATION DU CLIENT MCP : LECTURE DE RESSOURCE ===")
    resource_content = await handle_read_resource("enterprise://policies/severity-rules")
    print(f"Contenu récupéré :\n{resource_content}\n")
    
    print("=== SIMULATION DU CLIENT MCP : INVOCATION D'OUTILS ===")
    # Simulation d'un appel valide généré par le LLM
    valid_call_args = {"title": "Alerte OOM sur cluster production K8s", "severity": "CRITICAL", "assignee": "Ops-Team"}
    response = await handle_call_tool("create_new_incident", valid_call_args)
    print(f"Réponse du serveur (Appel Valide) : {response[0].text}\n")
    
    # Simulation d'un appel INVALIDE (Erreur de validation provoquée pour tester la robustesse)
    invalid_call_args = {"title": "Anomalie mineure UI", "severity": "LOW", "unknown_param": "hallucination"}
    print("Envoi de paramètres invalides au serveur...")
    failed_response = await handle_call_tool("create_new_incident", invalid_call_args)
    print(f"Réponse sécurisée du serveur (Appel Invalide) : {failed_response[0].text}")

# Lancement de la boucle d'événements de test
await simulate_client_discovery_and_execution()

## 💡 5. Notes de l'Architecte : Préparation à l'Intégration OpenAI & SDK Tiers

Lorsque vous interconnectez ce serveur MCP avec un orchestrateur moderne :
1. **Mappage vers OpenAI Tools** : Les données renvoyées par `list_tools` s'alignent directement sur le format `tools=[{"type": "function", "function": {...}}]` requis par les API OpenAI Chat Completions ou l'Assistant API. Le client MCP se charge d'adapter le formalisme.
2. **State Management** : Contrairement aux architectures REST stateless traditionnelles, une session MCP maintient un contexte d'échange. Veillez à ce que vos implémentations de serveurs d'outils n'accumulent pas d'état persistant instable en mémoire vive s'ils sont déployés derrière un répartiteur de charge (Load Balancer).

## 🛠️ 6. Exercice Guidé : Extension du Serveur MCP

**Votre objectif d'ingénierie backend :**
Modifiez ou enrichissez le code du serveur ci-dessus (vous pouvez réimplémenter ou surcharger les fonctions dans la cellule de réponse ci-dessous) pour ajouter un nouvel outil nommé `resolve_incident`.

**Spécifications de l'outil :**
1. Il doit accepter un argument obligatoire : `incident_id` (ex: `INC-001`).
2. Si l'ID existe dans `INCIDENT_DATABASE`, changez son statut à `"RESOLVED"`.
3. Si l'ID n'existe pas, l'outil doit retourner un message d'erreur clair et gracieusement formaté sans lever d'exception non gérée par le serveur.

### 📝 [Espace pour votre réponse - Exercice Guidé]
*Double-cliquez sur cette cellule pour implémenter votre code de validation Pydantic et votre branche de routage dans `handle_call_tool`.*

In [ ]:
# Votre code Python de résolution ici (Définition du schéma de validation + test d'appel)
class ResolveIncidentArgs(BaseModel):
    # À compléter
    pass

print("Prêt pour votre implémentation d'outils complémentaires.")